# Reproducible Science — a claim, end to end

Run the cells in order. Each one is the real command-line tool, not a simulation.

By the end you will have a manuscript claim bound to a run, the run bound to its outputs,
the outputs bound to their bytes — and you will have broken each of those links in turn to
see what the verifier says about it.

Nothing here needs a GPU, an account, or more than a few seconds.

## Install

In [ ]:
%pip install --quiet reproducible-science==0.2.0
!repro --version 2>/dev/null || echo 'installed'

## A project

A git repository, because a preregistration that names no commit is not anchored to
anything. `prereg freeze` refuses to run without one.

In [ ]:
import os
import pathlib
import subprocess
import textwrap

work = pathlib.Path("/tmp/demo")
subprocess.run(["rm", "-rf", str(work)])
work.mkdir(parents=True)
os.chdir(work)

!git init -q && git config user.email you@example.com && git config user.name You

pathlib.Path("analysis.py").write_text(
    textwrap.dedent("""
    import json, random
    random.seed(0)
    xs = [random.gauss(0.4, 0.1) for _ in range(200)]
    json.dump({"primary": {"effect": round(sum(xs) / len(xs), 3), "n": len(xs)}},
              open("results.json", "w"), indent=2)
""").strip()
)
print(pathlib.Path("analysis.py").read_text())

## 1. Freeze the plan

Before the analysis runs. The freeze records a digest of the plan and the commit it sat on,
so a plan edited afterwards is detectable.

In [ ]:
!prereg new study >/dev/null && cp study/PREREG.md . && rm -rf study
!git add -A && git commit -qm 'the plan'
!prereg freeze

## 2. Seal the inputs

Hashed *before* the run. Sealing afterwards proves nothing about what went in.

In [ ]:
!results init >/dev/null
!results seal PREREG.md analysis.py --role input

## 3. Run it, and record the output

In [ ]:
!python analysis.py
!cat results.json
!results run results.json --run-id exp_001 --note 'primary analysis'

## 4. Bind the manuscript claim to the run

In [ ]:
!results claim 'The primary effect was 0.404' --run-id exp_001 --location 'Table 2'

## 5. Declare what the manuscript says, and check it

The manifest is the only thing a reader needs in order to re-run this check.

In [ ]:
import json
import pathlib

pathlib.Path("repro.yaml").write_text(
    """
schema_version: repro/1
project: demo
artifacts:
  - id: results
    path: results.json
claims:
  - id: primary
    text: The primary effect was 0.404
    evidence:
      - kind: metric
        artifact: results
        name: effect
        pointer: /primary/effect
        reported: "0.404"
""".strip()
)
!repro verify repro.yaml

---
# Now break it

Each cell below breaks exactly one link in the chain. The point is not that the tool says
*no* — it is which kind of *no* it says.

## The manuscript prints a different number

The artifact was read and it disagrees. That is a `mismatch`, and it is a fact about the paper.

In [ ]:
text = pathlib.Path("repro.yaml").read_text().replace('"0.404"', '"0.410"')
pathlib.Path("repro.yaml").write_text(text)
!repro verify repro.yaml; echo "exit: $?"
pathlib.Path("repro.yaml").write_text(text.replace('"0.410"', '"0.404"'))

## The address resolves to nothing

Not a mismatch. Nothing was compared, so nothing disagreed — silence is not contradiction.

In [ ]:
text = pathlib.Path("repro.yaml").read_text().replace("/primary/effect", "/primary/effect_size")
pathlib.Path("repro.yaml").write_text(text)
!repro verify repro.yaml; echo "exit: $?"
pathlib.Path("repro.yaml").write_text(text.replace("/primary/effect_size", "/primary/effect"))

## The artifact changed after it was pinned

Pin it, edit it, and check again. Every number still agrees — with a document nobody pinned.
The comparison still runs and its result is still reported, marked non-authoritative.

In [ ]:
!repro pin repro.yaml >/dev/null 2>&1 || true
!repro verify repro.yaml --policy strict; echo "clean exit: $?"

d = json.load(open("results.json"))
d["primary"]["n"] = 201
json.dump(d, open("results.json", "w"), indent=2)
print("\n--- after editing a field the manuscript never cites ---\n")
!repro verify repro.yaml --policy strict; echo "exit: $?"

## The ledger was tampered with

The run record is a hash chain with a length anchor. Deleting an entry and re-anchoring is
the cheapest tampering there is, so `reanchor` refuses a chain it can see was truncated.

In [ ]:
!results verify

lines = pathlib.Path(".results/ledger.jsonl").read_text().splitlines()
pathlib.Path(".results/ledger.jsonl").write_text("\n".join(lines[:-1]) + "\n")
print("\n--- after deleting the last entry ---\n")
!results verify; echo "exit: $?"
print("\n--- trying to launder it ---\n")
!results reanchor; echo "exit: $?"

---
## What this showed

| break | verdict | why it matters |
|---|---|---|
| manuscript prints a different number | `mismatch` | the artifact was read and disagrees |
| address resolves to nothing | `not_found` | nothing was compared, so nothing disagreed |
| artifact edited after pinning | `broken_pin` | every number agrees, with the wrong document |
| ledger entry deleted | refused | truncation is visible, and cannot be re-anchored away |

A contradicted number and a missing tool are both failures of a build. They are not the same
fact about a paper, and the whole point of the contract is that the report says which one it
found.

Docs: <https://github.com/elliottower/reproducible-science>